# Milestone 3 - Decomposicao da Serie Temporal

Grupo 2

João Costa | 113564

Sílvia Ribeiro | 129428

Vitória Rodrigues | 130557

Este notebook prepara a etapa inicial de modelacao atraves da **decomposicao da serie temporal** para os quatro POIs considerados no projeto:

- Sete Cidades - Ponte entre Lagoas
- Vista do Rei - Miradouro
- Lagoa do Canario
- Caldeiras das Furnas

Os objetivos principais sao:

- carregar o dataset `df_preprocessed_eda.csv` a partir da pasta atual do notebook
- definir a meta de desempenho do modelo final: **MAPE <= 20%**
- calcular a capacidade maxima historica de cada POI
- criar a metrica `occupancy_rate`
- decompor a serie diaria de `total_incoming` em **tendencia, sazonalidade e residuo** com um modelo aditivo
- validar se os residuos se comportam como **ruido branco**, com recurso ao teste de Ljung-Box e ao grafico de autocorrelacao (ACF)
- apenas sugerir o treino do modelo CatBoost se os residuos passarem nos criterios de validacao estatistica

**Nota metodologica:** como os dados diarios por POI apresentam lacunas entre datas, a serie e regularizada para frequencia diaria apenas para fins de decomposicao, recorrendo a interpolacao temporal.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Configuracao do ambiente e criterio de sucesso

O notebook procura automaticamente a pasta central `dataset` a partir da diretoria atual de execução do Jupyter, onde acede ao ficheiro `df_preprocessed_eda.csv` gerado na Milestone 2.

O critério de sucesso definido para a etapa futura de modelação é:
- **MAPE máximo permitido = 20%**

In [ ]:
START_DIR = Path.cwd().resolve()
print(f"Diretoria atual de execução: {START_DIR}")

DATA_DIR = None
for root in [START_DIR, *START_DIR.parents]:
    candidate = root / "dataset"
    if candidate.exists() and candidate.is_dir():
        DATA_DIR = candidate.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Não foi possível localizar a pasta central 'dataset'. "
        "Certifica-te de que manténs a pasta 'dataset' na raiz do projeto."
    )

print(f"Pasta unificada de dados encontrada em: {DATA_DIR}")

DATA_PATH = DATA_DIR / "df_preprocessed_eda.csv"
TARGET_MAX_MAPE = 0.20

print(f"Caminho esperado do dataset: {DATA_PATH}")
print(f"MAPE máximo permitido para o modelo final: {TARGET_MAX_MAPE:.0%}")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Não foi encontrado o ficheiro 'df_preprocessed_eda.csv' em {DATA_PATH}. "
        "Certifica-te de que executaste o Notebook 2 (Pré-processamento)."
    )

## 2. Carregamento e inspecao inicial dos dados

Nesta fase validamos a estrutura do dataset e filtramos apenas os POIs definidos para o estudo de decomposicao temporal.

In [ ]:
df = pd.read_csv(DATA_PATH)
df["movement_date"] = pd.to_datetime(df["movement_date"], errors="coerce")

def inspect_dataframe(df, name, head_rows=5):
    print(f"{name} - shape: {df.shape}")
    display(df.head(head_rows))
    df.info()
    print("\nValores em falta (%):")
    display((df.isna().mean().sort_values(ascending=False) * 100).round(2).to_frame("missing_pct"))

inspect_dataframe(df, "Dataset pre-processado")


In [ ]:
TARGET_POIS = {
    9246: "Sete Cidades - Ponte entre Lagoas",
    10026: "Vista do Rei - Miradouro",
    7175: "Lagoa do Canario",
    6967: "Caldeiras das Furnas",
}

df_poi = df[df["geography_key"].isin(TARGET_POIS.keys())].copy()
df_poi["poi_label"] = df_poi["geography_key"].map(TARGET_POIS)
df_poi = df_poi.sort_values(["poi_label", "movement_date"]).reset_index(drop=True)

print("POIs presentes no dataset filtrado:")
display(df_poi[["geography_key", "poi_name", "poi_label"]].drop_duplicates().sort_values("geography_key"))
print("Cobertura temporal global:", df_poi["movement_date"].min().date(), "a", df_poi["movement_date"].max().date())
print("Numero de observacoes por POI:")
display(df_poi["poi_label"].value_counts().rename_axis("poi_label").to_frame("n_observacoes"))

## 3. Capacidade maxima historica e occupancy rate

A capacidade maxima de cada POI e definida como o maior valor historico observado em `total_incoming`.

A metrica `occupancy_rate` e calculada por:

`occupancy_rate = total_incoming / poi_capacity_max`

Esta variavel permite comparar o nivel relativo de ocupacao entre POIs com diferentes escalas absolutas de afluencia.

In [ ]:
capacity_by_poi = (
    df_poi.groupby(["geography_key", "poi_label"], dropna=False)["total_incoming"]
          .max()
          .rename("poi_capacity_max")
          .reset_index()
          .sort_values("poi_capacity_max", ascending=False)
)

df_poi = df_poi.merge(
    capacity_by_poi[["geography_key", "poi_capacity_max"]],
    on="geography_key",
    how="left",
    validate="m:1"
)
df_poi["occupancy_rate"] = df_poi["total_incoming"] / df_poi["poi_capacity_max"]

print("Capacidade maxima historica por POI:")
display(capacity_by_poi)

print("Amostra com occupancy_rate:")
display(
    df_poi[["movement_date", "geography_key", "poi_label", "total_incoming", "poi_capacity_max", "occupancy_rate"]].head(10)
)

## 4. Preparacao da serie diaria para decomposicao

A decomposicao aditiva exige uma serie temporal regular. Como existem dias sem observacao em alguns POIs, cada serie e reindexada para frequencia diaria e os valores em falta de `total_incoming` sao preenchidos por interpolacao temporal.

A periodicidade adotada e semanal (`period = 7`), por ser a sazonalidade diaria mais natural neste contexto de visitas turisticas.

In [ ]:
DECOMPOSITION_PERIOD = 7

prepared_series = {}
series_audit = []

for geo_key, poi_name in TARGET_POIS.items():
    poi_df = df_poi[df_poi["geography_key"] == geo_key].copy()
    poi_df = poi_df.sort_values("movement_date")

    full_index = pd.date_range(
        start=poi_df["movement_date"].min(),
        end=poi_df["movement_date"].max(),
        freq="D"
    )

    series = (
        poi_df.set_index("movement_date")["total_incoming"]
              .reindex(full_index)
              .interpolate(method="time")
              .ffill()
              .bfill()
    )

    prepared_series[geo_key] = {
        "poi_label": poi_name,
        "series": series,
    }

    series_audit.append({
        "geography_key": geo_key,
        "poi_label": poi_name,
        "original_observations": len(poi_df),
        "daily_observations_after_reindex": len(series),
        "start_date": series.index.min().date(),
        "end_date": series.index.max().date(),
        "missing_days_filled": int(series.index.size - poi_df["movement_date"].nunique()),
    })

series_audit_df = pd.DataFrame(series_audit)
display(series_audit_df)

## 5. Decomposicao aditiva por POI

Nesta etapa realizamos a decomposicao da serie temporal para cada POI, separando:

- **Tendencia**
- **Sazonalidade**
- **Residuo**

O objetivo e perceber se existe estrutura temporal suficientemente clara para justificar a modelacao supervisionada posterior.

In [ ]:
decomposition_results = {}

for geo_key, item in prepared_series.items():
    poi_label = item["poi_label"]
    series = item["series"]

    decomposition = seasonal_decompose(
        series,
        model="additive",
        period=DECOMPOSITION_PERIOD,
        extrapolate_trend="freq"
    )

    decomposition_results[geo_key] = {
        "poi_label": poi_label,
        "series": series,
        "decomposition": decomposition,
    }

    fig = decomposition.plot()
    fig.set_size_inches(12, 8)
    fig.suptitle(f"Decomposicao aditiva - {poi_label}", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Validacao estatistica dos residuos

Para cada POI avaliamos se os residuos se aproximam de **ruido branco**, combinando tres criterios:

1. **Residuos pequenos** em relacao a serie original
2. **Teste de Ljung-Box** com `p-value > 0.05`, sugerindo ausencia de autocorrelacao serial significativa
3. **ACF dos residuos** com autocorrelacoes baixas nos primeiros lags

A recomendacao de avancar para o treino do CatBoost so sera dada se todos os POIs passarem nesta validacao.

In [ ]:
validation_rows = []

for geo_key, item in decomposition_results.items():
    poi_label = item["poi_label"]
    series = item["series"]
    decomposition = item["decomposition"]
    resid = decomposition.resid.dropna()

    ljung_box_lag = min(DECOMPOSITION_PERIOD, max(1, len(resid) // 2))
    ljung_box_df = acorr_ljungbox(resid, lags=[ljung_box_lag], return_df=True)
    ljung_box_pvalue = float(ljung_box_df["lb_pvalue"].iloc[0])

    acf_lags = min(DECOMPOSITION_PERIOD, len(resid) - 1)
    acf_values = acf(resid, nlags=acf_lags, fft=False)
    max_abs_acf = float(np.max(np.abs(acf_values[1:]))) if len(acf_values) > 1 else 0.0

    series_std = float(series.std()) if float(series.std()) != 0 else np.nan
    series_mean = float(series.mean()) if float(series.mean()) != 0 else np.nan

    residual_std_ratio = float(resid.std() / series_std) if pd.notna(series_std) else np.nan
    residual_mean_abs_ratio = float(resid.abs().mean() / series_mean) if pd.notna(series_mean) else np.nan

    pass_small_residuals = (residual_std_ratio <= 0.50) and (residual_mean_abs_ratio <= 0.35)
    pass_ljung_box = ljung_box_pvalue > 0.05
    pass_acf = max_abs_acf <= 0.30
    pass_all = pass_small_residuals and pass_ljung_box and pass_acf

    validation_rows.append({
        "geography_key": geo_key,
        "poi_label": poi_label,
        "residual_observations": len(resid),
        "residual_std_ratio": residual_std_ratio,
        "residual_mean_abs_ratio": residual_mean_abs_ratio,
        "ljung_box_lag": ljung_box_lag,
        "ljung_box_pvalue": ljung_box_pvalue,
        "max_abs_acf": max_abs_acf,
        "pass_small_residuals": pass_small_residuals,
        "pass_ljung_box": pass_ljung_box,
        "pass_acf": pass_acf,
        "pass_all_checks": pass_all,
    })

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(resid.index, resid.values, color="#C44E52", marker="o")
    axes[0].axhline(0, color="black", linewidth=1, linestyle="--")
    axes[0].set_title(f"Residuos - {poi_label}")
    axes[0].set_xlabel("Data")
    axes[0].set_ylabel("Residuo")
    axes[0].tick_params(axis="x", rotation=30)

    plot_acf(resid, lags=acf_lags, ax=axes[1], title=f"ACF dos residuos - {poi_label}")
    plt.tight_layout()
    plt.show()

validation_summary = pd.DataFrame(validation_rows).sort_values("poi_label").reset_index(drop=True)
display(validation_summary.round(4))

## 7. Decisao sobre o avancar para modelacao

O treino do CatBoost so e recomendado quando os residuos se mostram suficientemente pequenos e sem padrao estatistico relevante.

Mesmo que a recomendacao seja positiva, o modelo final so sera aceite se cumprir:

- **MAPE <= 20%**

In [ ]:
all_pois_passed = bool(validation_summary["pass_all_checks"].all())

print(f"Meta de desempenho para o modelo final: MAPE <= {TARGET_MAX_MAPE:.0%}")
print(f"Todos os POIs passaram na validacao dos residuos? {all_pois_passed}")

if all_pois_passed:
    print(
        "\nConclusao: os residuos passaram na validacao estatistica. "
        "A serie apresenta condicoes para avancar para o treino de um modelo de regressao, "
        "incluindo a opcao CatBoost, desde que a avaliacao final mantenha MAPE <= 20%."
    )
else:
    print(
        "\nConclusao: os residuos nao passaram integralmente na validacao estatistica. "
        "Neste momento, o notebook nao sugere avancar diretamente para o treino do CatBoost. "
        "Antes disso, ampliar o historico temporal ou ajustar a estrategia de modelacao temporal."
    )


## 8. Exportacao opcional do resumo de validacao

A tabela resumo pode ser guardada para apoiar a discussao metodologica do Milestone 3 e a decisao sobre a etapa seguinte de modelacao.

In [ ]:
OUTPUT_PATH = DATA_DIR / "decomposition_validation_summary.csv"

validation_summary.to_csv(OUTPUT_PATH, index=False)

print(f"Resumo da validação estatística exportado com sucesso para:")
print(OUTPUT_PATH)